# 📘 Módulo 04 - Notebook 02: Imputación Estadística Avanzada

## 🎯 Imputación por Grupos y Métodos Avanzados

**Libro:** Saliendo de lo Pandito  
**Módulo:** 04 - Limpieza y Preparación de Datos  
**Duración estimada:** 65 minutos  
**Dificultad:** 🟠 Intermedio-Avanzado  
**Plataforma:** Databricks Free Edition

---

## 🎯 Objetivos

✅ **Dominar** imputación por grupos (GroupBy + transform)  
✅ **Aplicar** imputación condicional con .loc  
✅ **Implementar** métodos avanzados (KNN, iterativo)  
✅ **Evaluar** calidad de imputación  
✅ **Resolver** casos empresariales con múltiples estrategias

---

## 📚 Contenido

1. Limitaciones de Imputación Simple
2. Imputación por Grupos (GroupBy)
3. Imputación Condicional
4. Métodos Avanzados (KNN, MICE)
5. Evaluación de Calidad
6. Caso Integrador: Ventas por Región
7. Mejores Prácticas

---

## 💡 Por Qué Importa

**Imputación simple (promedio global) ignora estructura de datos.**

Ejemplo:
* 🌎 Ventas faltantes en "Sur": usar promedio **del Sur**, no promedio global
* 📊 Salarios faltantes: dependen de cargo, antigüedad, departamento
* 📅 Series temporales: valores cercanos en tiempo son más relevantes

**Imputación contextual = Datos más precisos = Decisiones mejores**

In [0]:
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

print("🎯 IMPUTACIÓN ESTADÍSTICA AVANZADA")
print("="*70)
print("• Imputación simple: Ignora estructura de datos")
print("• Imputación por grupos: Contexto por categorías")
print("• Métodos avanzados: KNN, MICE (multivariados)")
print("\n📖 Métodos clave:")
print("  - .groupby().transform()  : Imputar por grupos")
print("  - .loc[condicion]         : Imputación condicional")
print("  - KNNImputer              : K-vecinos más cercanos")
print("  - IterativeImputer        : MICE (multivariado)")
print("="*70)
print("✅ Librerías cargadas")

## ⚠️ El Problema de la Imputación Simple

### 🐛 Caso de Estudio: Ventas por Región

Imagina ventas faltantes en diferentes regiones:

```
Región  |  Ventas
--------|---------
Norte   |  50,000
Norte   |  52,000
Sur     |  ?
Sur     |  15,000
Sur     |  14,000
```

---

### 🚫 Imputación Incorrecta (Promedio Global)

```python
promedio_global = (50000 + 52000 + 15000 + 14000) / 4
= 32,750

Sur (?) = 32,750  # ¡INCORRECTO!
```

**Problema:** 32,750 NO representa ventas del Sur (promedio real ≈ 14,500)

---

### ✅ Imputación Correcta (Promedio por Grupo)

```python
promedio_sur = (15000 + 14000) / 2 = 14,500

Sur (?) = 14,500  # ✅ CORRECTO
```

**Solución:** Imputar usando **promedio del grupo** (región)

---

### 📊 Regla de Oro

👉 **Si los datos tienen estructura/grupos, SIEMPRE imputar por grupo**

Ejemplos:
* Ventas → por Región
* Salarios → por Cargo
* Precios → por Categoría
* Inventario → por Sucursal

In [0]:
import pandas as pd
import numpy as np

print("🔄 IMPUTACIÓN POR GRUPOS (GroupBy + Transform)")
print("="*70)

# Dataset de ventas por región con nulos
df = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=15),
    'Region': ['Norte', 'Norte', 'Norte', 'Norte', 'Norte',
               'Sur', 'Sur', 'Sur', 'Sur', 'Sur',
               'Centro', 'Centro', 'Centro', 'Centro', 'Centro'],
    'Vendedor': ['Ana', 'Luis', 'Ana', 'Luis', 'Ana',
                 'Pedro', 'Maria', 'Pedro', 'Maria', 'Pedro',
                 'Carlos', 'Sofia', 'Carlos', 'Sofia', 'Carlos'],
    'Ventas': [50000, 52000, np.nan, 51000, np.nan,
               15000, 14000, np.nan, 14500, 15500,
               32000, np.nan, 31000, 33000, np.nan]
})

print("\n📊 DataFrame original:")
print(df)
print(f"\nNulos en Ventas: {df['Ventas'].isnull().sum()}")

print("\n" + "-"*70)
print("\n1️⃣  PROMEDIO GLOBAL (INCORRECTO)")
promedio_global = df['Ventas'].mean()
df_global = df.copy()
df_global['Ventas'] = df_global['Ventas'].fillna(promedio_global)
print(f"\nPromedio global: ${promedio_global:,.0f}")
print("\nResultado (observe los valores imputados):")
print(df_global[['Region', 'Ventas']])
print("\n⚠️  Problema: Sur imputado con $33,833 (promedio global)")
print("    pero el promedio real del Sur es ~$14,750")

print("\n" + "="*70)
print("\n2️⃣  PROMEDIO POR GRUPO (CORRECTO)")
print("\nPromedio por Región:")
print(df.groupby('Region')['Ventas'].mean())

print("\n🎯 Método: .groupby().transform()")
df_grupo = df.copy()
df_grupo['Ventas'] = df_grupo.groupby('Region')['Ventas'].transform(
    lambda x: x.fillna(x.mean())
)

print("\nResultado (valores imputados con promedio de su región):")
print(df_grupo[['Region', 'Ventas']])
print("\n✅ Sur imputado con ~$14,750 (promedio del Sur)")

print("\n" + "-"*70)
print("\n3️⃣  MEDIANA POR GRUPO (mejor para outliers)")
df_mediana = df.copy()
df_mediana['Ventas'] = df_mediana.groupby('Region')['Ventas'].transform(
    lambda x: x.fillna(x.median())
)
print("\nMediana por Región:")
print(df.groupby('Region')['Ventas'].median())
print("\nResultado:")
print(df_mediana[['Region', 'Ventas']])

print("\n" + "="*70)
print("✅ Imputación por grupos dominada")

In [0]:
import pandas as pd
import numpy as np

print("🎯 IMPUTACIÓN CONDICIONAL CON .LOC")
print("="*70)

# Dataset de empleados con salarios faltantes
df = pd.DataFrame({
    'Empleado': ['Ana', 'Luis', 'Pedro', 'Maria', 'Carlos', 'Sofia', 'Juan', 'Laura'],
    'Cargo': ['Analista', 'Analista', 'Gerente', 'Gerente', 'Analista', 'Director', 'Gerente', 'Director'],
    'Antiguedad': [2, 3, 5, 6, 1, 10, 4, 8],
    'Salario': [45000, np.nan, 85000, np.nan, 42000, 120000, np.nan, 115000]
})

print("\n📊 DataFrame original:")
print(df)
print(f"\nNulos en Salario: {df['Salario'].isnull().sum()}")

print("\n" + "="*70)
print("\n1️⃣  ESTRATEGIA: Imputar según Cargo")
print("\nPromedio de Salario por Cargo:")
promedios_cargo = df.groupby('Cargo')['Salario'].mean()
print(promedios_cargo)

print("\n" + "-"*70)
print("\n2️⃣  IMPUTACIÓN CONDICIONAL CON .LOC")
df_imputado = df.copy()

# Imputar Analistas
mask_analista = (df_imputado['Cargo'] == 'Analista') & (df_imputado['Salario'].isnull())
df_imputado.loc[mask_analista, 'Salario'] = promedios_cargo['Analista']
print(f"\nAnalistas imputados con: ${promedios_cargo['Analista']:,.0f}")

# Imputar Gerentes
mask_gerente = (df_imputado['Cargo'] == 'Gerente') & (df_imputado['Salario'].isnull())
df_imputado.loc[mask_gerente, 'Salario'] = promedios_cargo['Gerente']
print(f"Gerentes imputados con: ${promedios_cargo['Gerente']:,.0f}")

print("\nResultado:")
print(df_imputado)

print("\n" + "="*70)
print("\n3️⃣  IMPUTACIÓN CONDICIONAL COMPLEJA (múltiples condiciones)")

# Dataset más complejo
df2 = pd.DataFrame({
    'Producto': ['A', 'A', 'A', 'B', 'B', 'B', 'C', 'C'],
    'Categoria': ['Premium', 'Premium', 'Estándar', 'Premium', 'Estándar', 'Estándar', 'Premium', 'Estándar'],
    'Precio': [100, np.nan, 80, 150, np.nan, 120, np.nan, 90]
})

print("\nDataFrame con múltiples grupos:")
print(df2)

print("\nImputar por Producto Y Categoría:")
df2_imputado = df2.copy()
df2_imputado['Precio'] = df2_imputado.groupby(['Producto', 'Categoria'])['Precio'].transform(
    lambda x: x.fillna(x.mean())
)

print("\nResultado (imputado por combinación Producto + Categoría):")
print(df2_imputado)

print("\n" + "="*70)
print("✅ Imputación condicional dominada")

## 🧠 Métodos de Imputación Avanzados

### 1️⃣  KNN Imputer (K-Nearest Neighbors)

**Concepto:** Imputar usando valores de registros **similares** (vecinos cercanos)

**Cómo funciona:**
1. Encuentra K filas más similares (sin nulos en esa columna)
2. Promedia sus valores
3. Imputa con ese promedio

**Ventajas:**
* Captura relaciones entre variables
* No asume distribución
* Funciona bien con patrones complejos

**Desventajas:**
* Lento en datasets grandes
* Sensible a escala (requiere normalización)
* Requiere todas las columnas numéricas

---

### 2️⃣  MICE (Multiple Imputation by Chained Equations)

**Concepto:** Imputación **iterativa multivariada**

**Cómo funciona:**
1. Imputación inicial (promedio)
2. Para cada columna con nulos:
   - Predecir valores usando otras columnas
   - Actualizar imputación
3. Repetir hasta convergencia

**Ventajas:**
* Captura relaciones complejas
* Mejor que imputación simple
* Flexible (distintos modelos por columna)

**Desventajas:**
* Computacionalmente costoso
* Puede no converger
* Requiere selección de hiperparámetros

---

### 📊 Cuándo Usar Cada Método

| Método | Uso Ideal |
|--------|----------|
| **Promedio/Mediana** | Datos simples, <10% nulos, MCAR |
| **Moda** | Categóricos, MCAR |
| **GroupBy + Transform** | Datos con grupos claros |
| **ffill/bfill** | Series temporales |
| **KNN** | Relaciones complejas, dataset pequeño |
| **MICE** | Relaciones multivariadas, dataset grande |

In [0]:
import pandas as pd
import numpy as np
from sklearn.impute import KNNImputer
from sklearn.preprocessing import StandardScaler

print("🔬 KNN IMPUTER (K-Nearest Neighbors)")
print("="*70)

# Dataset con relaciones entre variables
np.random.seed(42)
df = pd.DataFrame({
    'Edad': [25, 30, np.nan, 40, 45, 35, np.nan, 50],
    'Experiencia': [2, 5, 8, 15, 20, 10, 12, 25],
    'Salario': [40000, 55000, np.nan, 90000, 105000, 70000, np.nan, 120000]
})

print("\n📊 DataFrame original:")
print(df)
print(f"\nNulos: {df.isnull().sum().sum()}")

print("\n" + "="*70)
print("\n1️⃣  NORMALIZACIÓN (crítico para KNN)")
print("\nKNN es sensible a escala - debemos normalizar:")

scaler = StandardScaler()
df_scaled = pd.DataFrame(
    scaler.fit_transform(df),
    columns=df.columns
)
print("\nDatos normalizados:")
print(df_scaled.head())

print("\n" + "-"*70)
print("\n2️⃣  APLICAR KNN IMPUTER (k=3 vecinos)")
imputer = KNNImputer(n_neighbors=3)
df_imputed_scaled = imputer.fit_transform(df_scaled)

print("\n" + "-"*70)
print("\n3️⃣  DESNORMALIZAR (volver a escala original)")
df_imputed = pd.DataFrame(
    scaler.inverse_transform(df_imputed_scaled),
    columns=df.columns
)

print("\nResultado final:")
print(df_imputed)

print("\n" + "-"*70)
print("\n📊 COMPARACIÓN CON PROMEDIO SIMPLE")

df_promedio = df.copy()
df_promedio['Edad'] = df_promedio['Edad'].fillna(df_promedio['Edad'].mean())
df_promedio['Salario'] = df_promedio['Salario'].fillna(df_promedio['Salario'].mean())

print("\nMétodo 1 - Promedio Simple:")
print(df_promedio[['Edad', 'Experiencia', 'Salario']])

print("\nMétodo 2 - KNN (usando vecinos similares):")
print(df_imputed[['Edad', 'Experiencia', 'Salario']])

print("\n👉 KNN considera la relación Edad-Experiencia-Salario")
print("   para imputar valores más precisos")

print("\n" + "="*70)
print("✅ KNN Imputer dominado")

In [0]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error

print("📊 EVALUACIÓN DE CALIDAD DE IMPUTACIÓN")
print("="*70)

# Dataset completo (sin nulos) para simular
np.random.seed(42)
df_real = pd.DataFrame({
    'Region': ['Norte']*5 + ['Sur']*5,
    'Ventas': [50000, 52000, 51000, 53000, 49000,
               15000, 14000, 14500, 15500, 14200]
})

print("\n📊 Dataset REAL (sin nulos):")
print(df_real)
print(f"\nPromedio Norte: ${df_real[df_real['Region']=='Norte']['Ventas'].mean():,.0f}")
print(f"Promedio Sur: ${df_real[df_real['Region']=='Sur']['Ventas'].mean():,.0f}")

print("\n" + "="*70)
print("\n1️⃣  SIMULACIÓN: Introducir nulos artificialmente")

df_test = df_real.copy()
# Hacer nulos en posiciones específicas
indices_nulos = [1, 6]
valores_reales = df_test.loc[indices_nulos, 'Ventas'].values.copy()
df_test.loc[indices_nulos, 'Ventas'] = np.nan

print(f"\nValores reales antes de borrar: {valores_reales}")
print("\nDataset con nulos artificiales:")
print(df_test)

print("\n" + "="*70)
print("\n2️⃣  MÉTODO 1: Promedio Global (MALO)")
df_global = df_test.copy()
promedio_global = df_global['Ventas'].mean()
df_global['Ventas'] = df_global['Ventas'].fillna(promedio_global)
valores_imputados_global = df_global.loc[indices_nulos, 'Ventas'].values

print(f"\nPromedio global: ${promedio_global:,.0f}")
print(f"Valores imputados: {valores_imputados_global}")
print(f"Valores reales:    {valores_reales}")

mae_global = mean_absolute_error(valores_reales, valores_imputados_global)
print(f"\n📊 MAE (Mean Absolute Error): ${mae_global:,.0f}")

print("\n" + "="*70)
print("\n3️⃣  MÉTODO 2: Promedio por Grupo (BUENO)")
df_grupo = df_test.copy()
df_grupo['Ventas'] = df_grupo.groupby('Region')['Ventas'].transform(
    lambda x: x.fillna(x.mean())
)
valores_imputados_grupo = df_grupo.loc[indices_nulos, 'Ventas'].values

print(f"\nValores imputados: {valores_imputados_grupo}")
print(f"Valores reales:    {valores_reales}")

mae_grupo = mean_absolute_error(valores_reales, valores_imputados_grupo)
print(f"\n📊 MAE (Mean Absolute Error): ${mae_grupo:,.0f}")

print("\n" + "="*70)
print("\n🏆 COMPARACIÓN FINAL")
print("-"*70)
print(f"{'Método':<30} {'MAE':>15}")
print("-"*70)
print(f"{'Promedio Global':<30} ${mae_global:>14,.0f}")
print(f"{'Promedio por Grupo':<30} ${mae_grupo:>14,.0f}")
print("-"*70)
mejora = ((mae_global - mae_grupo) / mae_global) * 100
print(f"\n✅ Mejora con GroupBy: {mejora:.1f}%")

print("\n" + "="*70)
print("✅ Evaluación completada")

In [0]:
import pandas as pd
import numpy as np

print("💼 CASO INTEGRADOR: VENTAS REGIONALES CON MÚLTIPLES ESTRATEGIAS")
print("="*70)

# Dataset realista con estructura compleja
np.random.seed(42)
df = pd.DataFrame({
    'Fecha': pd.date_range('2024-01-01', periods=20),
    'Region': ['Norte', 'Norte', 'Sur', 'Sur', 'Centro'] * 4,
    'Categoria': ['Premium', 'Estándar'] * 10,
    'Vendedor': ['Ana', 'Luis', 'Pedro', 'Maria', 'Carlos'] * 4,
    'Ventas': [52000, 48000, np.nan, 14500, 31000,
               51000, np.nan, 15000, np.nan, 32000,
               np.nan, 49000, 14000, 15500, np.nan,
               53000, 50000, np.nan, 14200, 33000]
})

print("\n📄 DATOS ORIGINALES (20 transacciones):")
print(df)
print(f"\nNulos en Ventas: {df['Ventas'].isnull().sum()} de {len(df)} ({df['Ventas'].isnull().sum()/len(df)*100:.1f}%)")

print("\n" + "="*70)
print("\n🔍 ANÁLISIS EXPLORATORIO")
print("-"*70)

print("\n1. Distribución de nulos por Región:")
nulos_region = df.groupby('Region')['Ventas'].apply(lambda x: x.isnull().sum())
print(nulos_region)

print("\n2. Promedio de Ventas por Región (sin nulos):")
promedios_region = df.groupby('Region')['Ventas'].mean()
print(promedios_region)

print("\n3. Promedio por Región + Categoría:")
promedios_multi = df.groupby(['Region', 'Categoria'])['Ventas'].mean()
print(promedios_multi)

print("\n" + "="*70)
print("\n🛠️ ESTRATEGIA DE IMPUTACIÓN")
print("-"*70)
print("\n✅ Decisión: Imputar por Región + Categoría")
print("   Razón: Las ventas dependen de ambas variables")

df_imputado = df.copy()
df_imputado['Ventas'] = df_imputado.groupby(['Region', 'Categoria'])['Ventas'].transform(
    lambda x: x.fillna(x.mean())
)

print("\n" + "="*70)
print("\n✅ RESULTADO FINAL")
print("-"*70)
print(df_imputado)

print("\n" + "-"*70)
print("\n📊 VALIDACIÓN POST-IMPUTACIÓN")
print("-"*70)
print(f"\n1. Nulos restantes: {df_imputado['Ventas'].isnull().sum()}")
print(f"\n2. Total de Ventas:")
print(f"   - Antes (con nulos): ${df['Ventas'].sum():,.0f}")
print(f"   - Después (imputado): ${df_imputado['Ventas'].sum():,.0f}")

print(f"\n3. Promedio por Región (post-imputación):")
print(df_imputado.groupby('Region')['Ventas'].mean())

print("\n4. Distribución de Ventas Imputadas:")
print(f"   - Norte Premium: ~${df_imputado[(df_imputado['Region']=='Norte') & (df_imputado['Categoria']=='Premium')]['Ventas'].mean():,.0f}")
print(f"   - Sur Estándar: ~${df_imputado[(df_imputado['Region']=='Sur') & (df_imputado['Categoria']=='Estándar')]['Ventas'].mean():,.0f}")
print(f"   - Centro Premium: ~${df_imputado[(df_imputado['Region']=='Centro') & (df_imputado['Categoria']=='Premium')]['Ventas'].mean():,.0f}")

print("\n" + "="*70)
print("✅ Caso integrador completado con imputación contextual")

## 🎓 Conclusiones del Notebook 04_02

### ✅ Lo Que Aprendiste

1. **Limitaciones de Imputación Simple:**
   - Promedio global ignora estructura de datos
   - Puede introducir sesgo significativo
   - No captura relaciones entre variables

2. **Imputación por Grupos:**
   - `.groupby().transform(lambda x: x.fillna(x.mean()))`
   - Respeta estructura de datos
   - Mucho más precisa que promedio global

3. **Imputación Condicional:**
   - `.loc[condición, columna] = valor`
   - Control total sobre la estrategia
   - Útil para reglas de negocio complejas

4. **Métodos Avanzados:**
   - KNN: Usa vecinos similares
   - MICE: Imputación iterativa multivariada
   - Requieren normalización y más poder computacional

5. **Evaluación de Calidad:**
   - MAE (Mean Absolute Error)
   - Comparación de métodos
   - Validación con datos simulados

---

### 🎯 Reglas de Oro

👉 **Regla #1: Si hay grupos, imputar POR GRUPO**
```python
df.groupby('Grupo')['Columna'].transform(lambda x: x.fillna(x.mean()))
```

👉 **Regla #2: Validar SIEMPRE**
```python
# Antes
print(df.isnull().sum())
# Después
print(df_imputado.isnull().sum())
```

👉 **Regla #3: Documentar decisiones**
```python
# MALO: df['Ventas'].fillna(20000)  # ¿Por qué 20000?
# BUENO:
# Imputar con promedio regional porque las ventas
# varían 3x entre Norte y Sur
df.groupby('Region')['Ventas'].transform(lambda x: x.fillna(x.mean()))
```

---

### 📊 Guía de Decisión

| Situación | Método |
|-----------|--------|
| Datos con grupos claros | `.groupby().transform()` |
| Múltiples condiciones | `.loc[condición, col] = valor` |
| Relaciones complejas, dataset pequeño | KNNImputer |
| Relaciones multivariadas | MICE (IterativeImputer) |
| Series temporales | ffill / bfill / interpolate |

---

### 🚀 Próximo Notebook

**04_03 - Transformación de Cadenas de Texto**
* Limpieza de strings (espacios, mayúsculas)
* Extracción de patrones
* Split y Join
* Casos: RUC, emails, teléfonos

---

<div style="background: linear-gradient(90deg, #2563eb 0%, #60a5fa 100%); padding: 20px; border-radius: 10px; color: white; text-align: center;">
  <h3>🎯 ¡Imputación Avanzada Dominada!</h3>
  <p><i>"Imputar con contexto es la diferencia entre datos precisos y datos engañosos."</i></p>
</div>